In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, FloatType, DateType
import pyspark.sql.functions as F

In [0]:
CATALOG_NAME = 'fashion_retail'
BRONZE_DATASET = 'bronze'
DATA_SOURCE_PATH = '/Volumes/fashion_retail/source_data/fashion_retail_raw_data'

# Define schema for the data file
stores_schema = StructType([
    StructField('StoreID', IntegerType(), True),
    StructField('Country', StringType(), True),
    StructField('City', StringType(), True),
    StructField('StoreName', StringType(), True),
    StructField('NumberOfEmployees', IntegerType(), True),
    StructField('ZIPCode', StringType(), True),
    StructField('Latitude', FloatType(), True),
    StructField('Longitude', FloatType(), True),
])

employees_schema = StructType([
    StructField('EmployeeID', IntegerType(), True),
    StructField('StoreID', IntegerType(), True),
    StructField('Name', StringType(), True),
    StructField('Position', StringType(), True),
])

products_schema = StructType([
    StructField('ProductID', IntegerType(), True),
    StructField('Category', StringType(), True),
    StructField('SubCategory', StringType(), True),
    StructField('DescriptionPT', StringType(), True),
    StructField('DescriptionDE', StringType(), True),
    StructField('DescriptionFR', StringType(), True),
    StructField('DescriptionES', StringType(), True),
    StructField('DescriptionEN', StringType(), True),
    StructField('DescriptionZH', StringType(), True),
    StructField('Color', StringType(), True),
    StructField('Sizes', StringType(), True),
    StructField('ProductionCost', FloatType(), True),
])

discounts_schema = StructType([
    StructField('Start', DateType(), True),
    StructField('End', DateType(), True),
    StructField('Discont', FloatType(), True),
    StructField('Description', StringType(), True),
    StructField('Category', StringType(), True),
    StructField('SubCategory', StringType(), True),
])

customers_schema = StructType([
    StructField('customerid', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('telephone', StringType(), True),
    StructField('city', StringType(), True),
    StructField('country', StringType(), True),
    StructField('gender', StringType(), True),
    StructField('dateofbirth', DateType(), True),
    StructField('jobtitle', StringType(), True),
])

transactions_schema = StructType([
    StructField('invoiceid', StringType(), True),
    StructField('line', IntegerType(), True),
    StructField('customerid', IntegerType(), True),
    StructField('productid', IntegerType(), True),
    StructField('size', StringType(), True),
    StructField('color', StringType(), True),
    StructField('unitprice', FloatType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('date', TimestampType(), True),
    StructField('discount', FloatType(), True),
    StructField('linetotal', FloatType(), True),
    StructField('storeid', IntegerType(), True),
    StructField('employeeid', IntegerType(), True),
    StructField('currency', StringType(), True),
    StructField('currencysymbol', StringType(), True),
    StructField('sku', StringType(), True),
    StructField('transactiontype', StringType(), True),
    StructField('paymentmethod', StringType(), True),
    StructField('invoicetotal', FloatType(), True),
])


### Load Store Data

In [0]:

store_raw_data = f"{DATA_SOURCE_PATH}/stores.csv"

df = spark.read.option("header", "true").option("delimeter", ",").schema(stores_schema).csv(store_raw_data)

# Add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

# display(df.limit(5))

### Write Raw Store Data to Bronze

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .saveAsTable(f"{CATALOG_NAME}.{BRONZE_DATASET}.stores")

Load Employees Data

In [0]:
employees_raw_data = f"{DATA_SOURCE_PATH}/employees.csv"

df = spark.read.option("header", "true").option("delimeter", ",").schema(employees_schema).csv(employees_raw_data)

# Add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())


### Write Raw Employees Data to Bronze

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .saveAsTable(f"{CATALOG_NAME}.{BRONZE_DATASET}.employees")

### Load Products Data

In [0]:
products_raw_data = f"{DATA_SOURCE_PATH}/products.csv"

df = spark.read.option("header", "true").option("delimeter", ",").schema(products_schema).csv(products_raw_data)

# Add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

### Write Raw Products Data to Bronze

In [0]:
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .saveAsTable(f"{CATALOG_NAME}.{BRONZE_DATASET}.products")

### Read and Write Customers Data

In [0]:
customers_raw_data = f"{DATA_SOURCE_PATH}/customers/*.csv"

df = spark.read.option("header", "true").option("delimeter", ",").schema(customers_schema).csv(customers_raw_data)

# Add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

# display(df.limit(5))

In [0]:
# WRITE CUSTOMERS TO DELTA TABLE (BRONZE)
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .saveAsTable(f"{CATALOG_NAME}.{BRONZE_DATASET}.customers")

### Read and Write Transactions Data

In [0]:
transaction_raw_data = f"{DATA_SOURCE_PATH}/transactions/*.csv.gz"

df = spark.read.option("header", "true").option("delimeter", ",").schema(transactions_schema).csv(transaction_raw_data)

# Add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

display(df.count())


In [0]:
# WRITE TRANSACTIONS TO DELTA TABLE (BRONZE)
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .saveAsTable(f"{CATALOG_NAME}.{BRONZE_DATASET}.transactions")

### Read and Write Discount Data

In [0]:
discounts_raw_data = f"{DATA_SOURCE_PATH}/discounts.csv"

df = spark.read.option("header", "true").option("delimeter", ",").schema(discounts_schema).csv(discounts_raw_data)

# Add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

display(df.count())

In [0]:
# WRITE DISCOUNTS TO DELTA TABLE (BRONZE)
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema", "true") \
  .saveAsTable(f"{CATALOG_NAME}.{BRONZE_DATASET}.discounts")